In [3]:
import numpy as np
import sys
import os

sys.path.append(os.path.abspath("../src"))

from src.coils import coils
from src.pmts import PMTs

In [4]:
import numpy as np

Ax = np.load("../data/Ax.npy")
Ay = np.load("../data/Ay.npy")
Az = np.load("../data/Az.npy")
A  = np.load("../data/A.npy")

print(A.shape)

(18, 7)


In [5]:
#ambient field
B_ambient = np.array([
    -283.38,
    106.79,
    -381.21
])

In [6]:
target_single = -B_ambient

In [7]:
Bx_target = np.full(6, target_single[0])
By_target = np.full(6, target_single[1])
Bz_target = np.full(6, target_single[2])

B_target = np.concatenate([
    Bx_target,
    By_target,
    Bz_target
])

In [8]:
B_target.shape

(18,)

In [9]:
I_opt, residuals, rank, s = np.linalg.lstsq(
    A,
    B_target,
    rcond=None
)

In [10]:
coil_names = ['X1','X2','X3','Y1','Y2','Z1','Z2']

print("Optimized currents:\n")

for name, current in zip(coil_names, I_opt):

    print(f"{name}: {current:.2f} A")

Optimized currents:

X1: 57.14 A
X2: 34.18 A
X3: 69.61 A
Y1: -21.24 A
Y2: -26.87 A
Z1: 77.58 A
Z2: 66.06 A


In [11]:
# Columns:
# 0 -> X1
# 1 -> X2
# 2 -> X3
# 3 -> Y1
# 4 -> Y2
# 5 -> Z1
# 6 -> Z2

A_red = np.column_stack([

    A[:,0],                 # X1
    A[:,1],                 # X2
    A[:,2],                 # X3

    A[:,3] + A[:,4],        # Y grouped

    A[:,5] + A[:,6]         # Z grouped
])

print(A_red.shape)

(18, 5)


In [12]:
I_red, residuals, rank, s = np.linalg.lstsq(
    A_red,
    B_target,
    rcond=None
)

In [13]:
labels = ['X1','X2', 'X3','Y_group','Z_group']

print("Optimized grouped currents:\n")

for name, current in zip(labels, I_red):

    print(f"{name}: {current:.2f} A")

Optimized grouped currents:

X1: 60.99 A
X2: 33.36 A
X3: 68.45 A
Y_group: -23.20 A
Z_group: 71.84 A


In [14]:
I_full = np.array([

    I_red[0],   # X1
    I_red[1],   # X2
    I_red[2],   # X3

    I_red[3],   # Y1
    I_red[3],   # Y2 same

    I_red[4],   # Z1
    I_red[4]    # Z2 same
])

In [15]:
B_pred = A @ I_full

In [16]:
residual = B_pred - B_target
print(residual)

[-6.30359634 -0.36696595  0.1700823  -0.79558928 13.82545228 -7.36671683
 -7.21493241 -1.7804436   4.11201656 -1.63513503 -1.08553789  9.6941661
  3.66039976 -2.07233434  1.38542654  1.00244429 -5.3309821   1.23443539]


In [17]:
print(B_pred)

[ 277.07640366  283.01303405  283.5500823   282.58441072  297.20545228
  276.01328317 -114.00493241 -108.5704436  -102.67798344 -108.42513503
 -107.87553789  -97.0958339   384.87039976  379.13766566  382.59542654
  382.21244429  375.8790179   382.44443539]


In [18]:
# ============================================================
# Reduced matrix with grouped currents
#
# Columns:
# 0 -> X_outer = X1 + X3
# 1 -> X2
# 2 -> Y_group = Y1 + Y2
# 3 -> Z_group = Z1 + Z2
# ============================================================

A_red2 = np.column_stack([

    A[:,0] + A[:,2],     # X1 + X3

    A[:,1],              # X2

    A[:,3] + A[:,4],     # Y group

    A[:,5] + A[:,6]      # Z group
])

print("A_red2 shape =", A_red2.shape)

I_red2, residuals, rank, s = np.linalg.lstsq(
    A_red2,
    B_target,
    rcond=None
)

labels = [
    'X_outer',
    'X2',
    'Y_group',
    'Z_group'
]

print("Optimized grouped currents:\n")

for name, current in zip(labels, I_red2):

    print(f"{name}: {current:.2f} A")



I_full2 = np.array([

    I_red2[0],   # X1
    I_red2[1],   # X2
    I_red2[0],   # X3

    I_red2[2],   # Y1
    I_red2[2],   # Y2 same

    I_red2[3],   # Z1
    I_red2[3]    # Z2 same
])

B_pred2 = A @ I_full2

residual2 = B_pred2 - B_target
print(residual2)

A_red2 shape = (18, 4)
Optimized grouped currents:

X_outer: 62.44 A
X2: 34.80 A
Y_group: -23.12 A
Z_group: 71.85 A
[ -1.55189857   4.1303597    4.74259247  -5.02663181   8.31619379
 -11.57432681  -7.77735747  -1.59006871   5.44316824  -6.55377686
  -1.35527942  15.34569305   3.70401979  -2.0291292    1.43053744
   0.96221458  -5.38246438   1.19337841]


In [19]:
import numpy as np

# ============================================================
# TABLE 6 currents
# ============================================================

I_table6 = np.array([

    68,   # X1
    32,   # X2
    68,   # X3

    -23,   # Y1
    -23,   # Y2

    72,   # Z1
    72    # Z2

])

# ============================================================
# Ambient field from Table 4
# ============================================================

Bx_ambient = -283.38
By_ambient =  106.79
Bz_ambient = -381.21

# ============================================================
# Build ambient field vector for all PMTs
# ============================================================

B_ambient = np.array([

    # Bx block
    Bx_ambient,
    Bx_ambient,
    Bx_ambient,
    Bx_ambient,
    Bx_ambient,
    Bx_ambient,

    # By block
    By_ambient,
    By_ambient,
    By_ambient,
    By_ambient,
    By_ambient,
    By_ambient,

    # Bz block
    Bz_ambient,
    Bz_ambient,
    Bz_ambient,
    Bz_ambient,
    Bz_ambient,
    Bz_ambient

])

# ============================================================
# Compensation field from coils
# ============================================================

B_comp = A @ I_table6

# ============================================================
# Final residual field
# ============================================================

B_final = B_ambient + B_comp

# ============================================================
# Print PMT-wise results
# ============================================================

for i in range(6):

    Bx = B_final[i]
    By = B_final[i+6]
    Bz = B_final[i+12]

    Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)

    print(f"\nPMT{i+1}")
    print("-"*30)

    print(f"Bx   = {Bx:.2f} mG")
    print(f"By   = {By:.2f} mG")
    print(f"Bz   = {Bz:.2f} mG")
    print(f"|B|  = {Bmag:.2f} mG")


PMT1
------------------------------
Bx   = 5.89 mG
By   = -12.73 mG
Bz   = 4.42 mG
|B|  = 14.71 mG

PMT2
------------------------------
Bx   = 12.71 mG
By   = -1.58 mG
Bz   = -1.34 mG
|B|  = 12.88 mG

PMT3
------------------------------
Bx   = 7.01 mG
By   = 10.61 mG
Bz   = 2.16 mG
|B|  = 12.89 mG

PMT4
------------------------------
Bx   = -2.71 mG
By   = -1.33 mG
Bz   = 1.87 mG
|B|  = 3.55 mG

PMT5
------------------------------
Bx   = 12.83 mG
By   = -0.33 mG
Bz   = -4.47 mG
|B|  = 13.59 mG

PMT6
------------------------------
Bx   = -9.23 mG
By   = 11.20 mG
Bz   = 2.10 mG
|B|  = 14.66 mG


In [20]:
import numpy as np


# ============================================================
# Ambient field from Table 4
# ============================================================

Bx_ambient = -283.38
By_ambient =  106.79
Bz_ambient = -381.21

# ============================================================
# Build ambient field vector for all PMTs
# ============================================================

B_ambient = np.array([

    # Bx block
    Bx_ambient,
    Bx_ambient,
    Bx_ambient,
    Bx_ambient,
    Bx_ambient,
    Bx_ambient,

    # By block
    By_ambient,
    By_ambient,
    By_ambient,
    By_ambient,
    By_ambient,
    By_ambient,

    # Bz block
    Bz_ambient,
    Bz_ambient,
    Bz_ambient,
    Bz_ambient,
    Bz_ambient,
    Bz_ambient

])

# ============================================================
# Compensation field from coils
# ============================================================

B_comp2 = A @ I_full

# ============================================================
# Final residual field
# ============================================================

B_final2 = B_ambient + B_comp2

# ============================================================
# Print PMT-wise results
# ============================================================

for i in range(6):

    Bx = B_final2[i]
    By = B_final2[i+6]
    Bz = B_final2[i+12]

    Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)

    print(f"\nPMT{i+1}")
    print("-"*30)

    print(f"Bx   = {Bx:.2f} mG")
    print(f"By   = {By:.2f} mG")
    print(f"Bz   = {Bz:.2f} mG")
    print(f"|B|  = {Bmag:.2f} mG")


PMT1
------------------------------
Bx   = -6.30 mG
By   = -7.21 mG
Bz   = 3.66 mG
|B|  = 10.26 mG

PMT2
------------------------------
Bx   = -0.37 mG
By   = -1.78 mG
Bz   = -2.07 mG
|B|  = 2.76 mG

PMT3
------------------------------
Bx   = 0.17 mG
By   = 4.11 mG
Bz   = 1.39 mG
|B|  = 4.34 mG

PMT4
------------------------------
Bx   = -0.80 mG
By   = -1.64 mG
Bz   = 1.00 mG
|B|  = 2.08 mG

PMT5
------------------------------
Bx   = 13.83 mG
By   = -1.09 mG
Bz   = -5.33 mG
|B|  = 14.86 mG

PMT6
------------------------------
Bx   = -7.37 mG
By   = 9.69 mG
Bz   = 1.23 mG
|B|  = 12.24 mG
